# Setup — Orientation: the map, the toolchain, and an environment check

*The on-ramp to the [`notebooks/` course](README.md) on computational synthetic morphology — **not a
numbered lab**: read this, run the env check, do the four hello-worlds, then go to **Lab 1**.*

This course is a sequence of **self-contained labs** that walk from *"what is a regulome and why a
hypergraph"* to *"design an intervention that steers a tissue toward a target state."* The arc, one
line each:

| # | one line | needs |
|---|---|---|
| **Setup** *(here)* | the map, the toolchain, the env check | `numpy`, `matplotlib`, `jax`, `scipy` |
| **Lab 1** [`01`](01_gene_circuit_dynamics.ipynb) | gene-circuit dynamics in this course's toolchain — Hill functions, negative autoregulation, the toggle switch & bistability, the repressilator, "linearise then LQR" — the *bridge from* Elowitz & Bois | `diffrax`, `jax`, (`jaxctrl`) |
| **Lab 2** [`02`](02_regulomes_and_hypergraphs.ipynb) | a regulon is a **hyperedge**, not a clique; the incidence matrix *is* the hypergraph; the Laplacian spectrum | `hgx` |
| **Lab 3** [`03`](03_benchmarking_fidelity.ipynb) | does the regulome **predict perturbations**? the fidelity *triple* (direction transfers, magnitude doesn't) | — |
| **Lab 4** [`04`](04_modularity_identifiability.ipynb) | does it **decompose into modules**? the Hodge Laplacian → the **Module Identifiability Index** | `scipy` |
| **Lab 5** [`05`](05_hypergraph_neural_odes.ipynb) | **learn the flow** — cell types as attractors; rollout-MSE splits stable drivers from transient stress | `jax`, `optax`, (`diffrax`/`hgx`) |
| **Lab 6** [`06`](06_control_theory.ipynb) | the regulome as a **steerable plant** — controllability, Gramians, LQR, driver nodes | `scipy`, `jax`, (`jaxctrl`) |
| **Lab 7** [`07`](07_structural_identifiability.ipynb) | is the *mechanistic* reduction's $\theta$ **recoverable** from perfect data? (the symbolic pre-condition) | `sympy` |
| **Lab 8** [`08`](08_anatomical_compiler.ipynb) | **the anatomical compiler** — direct optimal control on the learned ODE: target state in → actuation out | `jax`, `optax`, (`diffrax`/`jaxctrl`) |
| **Lab 9** [`09`](09_synthetic_morphology_wetlab.ipynb) | the **wet-lab programme** — synNotch · 4-D bioprinting · bioelectric · biobots — each one optimal-control problem on the ODE | — |
| **Lab 10** [`10`](10_cancer_module_identifiability.ipynb) | **cancer as loss of module identifiability** — the metrics turned diagnostic | `scipy` |

Companion material: the **`jaxctrl`** repo's three worked examples (`repressilator_control_demo.py`,
`irma_sindy_lqr.ipynb`, `grn_hypergraph_drivers.ipynb` — the nonlinear / SINDy-surrogate /
hypergraph-driver versions of Labs 6–8); [`organoid_hgx_colab.ipynb`](organoid_hgx_colab.ipynb) (the
GPU end-to-end run of the real Fleck-organoid pipeline — optional, needs a GPU; *unnumbered*); and the
**BETSE-JAX refactor** at `~/Workspace/betse-unified` — the bioelectric-layer companion (a
differentiable JAX rewrite of the BioElectric Tissue Simulation Engine; `betse.science.jax.inverse.optimize_pattern`
is the bioelectric instance of "the anatomical compiler" — target $V_{\rm mem}$ pattern in → required
ion currents out — and its bridge examples, inverse bioelectric design / xenobot motility,
$V_{\rm mem}$→GRN prepatterning, morphoceutical timelines, are the wet-lab partner to Labs 8–10).

The labs are self-contained (each reads the committed data or generates its own toy data) but they
*build* on each other conceptually — the dependency map is §4 below. Run order: **Setup → 1 → 2 → 3 →
4 → 5 → 6 → 7 → 8 → 9 → 10**, skipping Lab 1 if you've done Elowitz & Bois.

## 1. Prerequisites — what to know coming in

This course does *not* re-derive these; if a topic below is unfamiliar, do the named warm-up first.

- **Python + scientific computing** — `numpy`, `matplotlib`; reading and running notebooks. (If you
  use [`uv`](https://docs.astral.sh/uv/): `uv sync` in the repo root gives you everything.)
- **JAX — the differentiable-programming mindset** *(the load-bearing prerequisite — everything here
  is JAX)*: `jax.jit`, `jax.grad` (reverse-mode autodiff), `jax.vmap`, pure functions, pytrees,
  immutable arrays (`x.at[i].set(v)`), and `jax.lax.{scan,fori_loop,cond}` instead of Python control
  flow inside jitted code. Every optimisation in this course — fitting a Neural ODE (Lab 5), finding
  an actuation schedule (Lab 8), the SBI inverse (Lab 8), the inverse bioelectric design in BETSE-JAX
  — is `jax.grad` *through* something. If JAX is new: the JAX docs' "JAX in 100 seconds" / the Equinox
  intro, then come back. (§3 below gives a 5-line refresher.)
- **Linear algebra** — eigenvalues/eigenvectors, symmetric matrices, the **graph Laplacian** $L = D-A$
  (Labs 2, 4, 6 lean on its spectrum: the harmonic null space = connected components, the Fiedler
  value = algebraic connectivity, the eigengap = a module-count signal).
- **ODEs & dynamical systems** — fixed points, **linear stability** (the Jacobian's eigenvalues),
  **attractors** and basins, separatrices. The "cell type = attractor of the GRN dynamics" picture
  (Kauffman; Huang) is the spine of Labs 5–8 and 10.
- **Basic molecular / developmental biology** — what a *transcription factor* is, what a *gene
  regulatory network* is, what an *organoid* is, what *pseudotime* means. Light — the labs re-explain
  in passing; Davidson's *The Regulatory Genome* is the deep version (GRN *kernels* / differentiation
  gene batteries — Lab 2).
- **Elowitz & Bois, *Biological Circuit Design*** ([biocircuits.github.io](https://biocircuits.github.io/),
  Caltech BE 150 / Bi 250b) — the *dynamical-systems-of-gene-circuits* foundations (Hill kinetics,
  autoregulation, the toggle switch, the repressilator, ultrasensitivity, exact adaptation, stochastic
  expression, the classic patterning circuits). The **prerequisite/sibling** of this track;
  **[Lab 1](01_gene_circuit_dynamics.ipynb) is the explicit bridge from it** (same circuits, this
  course's `diffrax`/`jax` toolchain). A student who's done it can skip Lab 1.
- *(context, not a hard prereq)* **HTGAA 2026a Weeks 6–7** — *Genetic Circuits I/II* (DNA assembly +
  the Asimov Kernel platform; intracellular ANNs / perceptrons in cells) — the wet-lab-circuits world
  this track is the computational complement to.

You do **not** need prior `hgx` / `jaxctrl` / `diffrax` / `sympy` — §3 is your 5-minute intro to each,
and the labs reimplement the bits they need so they run without `jaxctrl`.

## 2. The data substrate — the organoid regulome

Most labs use one dataset as the running example: the **cerebral-organoid regulome of Fleck et al.
(2023, *Nature*)** — a genome-scale gene regulatory network *inferred from data* (single-cell multiome:
RNA + ATAC) with the **Pando** method. It comes to us as a binary **incidence matrix** $H\in\{0,1\}^{
\text{genes}\times\text{regulons}}$: $H_{g,r}=1$ iff gene $g$ is a member of regulon $r$ (regulon $r$ =
"TF $r$ and the set of genes it regulates"). Roughly **720 TFs / regulons**, **2,792 genes**, ~**75k**
TF→gene edges. It lives in `data/processed/` (`incidence.npy`, `gene_names.json`, `tf_names.json`,
plus `temporal_expression.npy`, `fate_probabilities.npy`, `perturbation_effects.npy`, …, and the
spectra / module labels). Lab 2 unpacks all of it; the env check below just loads it (or, if it's
absent, a tiny synthetic regulome — every lab degrades gracefully that way).

In [ ]:
# ---- ENV CHECK: if this cell runs without error, you're ready for Lab 1 ----------------------
import sys, json
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import jax, jax.numpy as jnp
import scipy

print("REQUIRED (every lab):")
for m in (np, matplotlib, jax, scipy):
    print(f"  {m.__name__:>12}  {getattr(m, '__version__', '?')}")
print(f"  python        {sys.version.split()[0]}   |   jax devices: {jax.devices()}")

print("\nRECOMMENDED / per-lab (soft — the labs reimplement what they need, but install for the full experience):")
for name, used_by in [("equinox", "Lab 5 (Neural ODE params)"), ("diffrax", "Labs 1/5/8 (ODE solvers + adjoints)"),
                      ("optax",   "Labs 5/8 (optimisers)"),       ("hgx",     "Labs 2/4/5 (hypergraph NNs)"),
                      ("sympy",   "Lab 7 (symbolic identifiability)"), ("jaxctrl", "Labs 6/8 + the example notebooks (control)"),
                      ("scanpy",  "the benchmark scripts (h5ad I/O)")]:
    try:
        mod = __import__(name); print(f"  {name:>12}  {getattr(mod, '__version__', 'ok'):<10} — {used_by}")
    except Exception:
        print(f"  {name:>12}  MISSING    — {used_by}   (uv sync / pip install {name})")

# ---- load the organoid regulome (or a tiny synthetic fallback) ----
def _find(*parts):
    here = Path.cwd()
    for base in [here, *here.parents]:
        p = base.joinpath(*parts)
        if p.exists():
            return p
    return None
PROC = _find("data", "processed")
if PROC is not None and (PROC / "incidence.npy").exists():
    H          = np.load(PROC / "incidence.npy").astype(np.float32)
    gene_names = json.loads((PROC / "gene_names.json").read_text())
    tf_names   = json.loads((PROC / "tf_names.json").read_text())
    src = "Fleck et al. 2023 cerebral-organoid regulome (Pando)"
else:
    print("\n[note] data/processed/ not found — using a tiny synthetic regulome (so this notebook still runs).")
    gene_names = [f"g{i}" for i in range(12)]; tf_names = [f"TF{j}" for j in range(4)]
    H = np.zeros((12, 4), np.float32)
    for j, mem in enumerate([[0,1,2,3,4],[3,4,5,6,7],[6,7,8,9],[1,2,8,10,11]]):
        for g in mem: H[g, j] = 1.0
    src = "synthetic toy regulome (12 genes, 4 regulons)"
print(f"\nloaded: {src}")
print(f"  genes (hypergraph nodes): {H.shape[0]:>6}   regulons (hyperedges): {H.shape[1]:>6}   TF→gene incidences: {int(H.sum()):>8,}")
print("  → Lab 2 takes it from here.  This is the object the whole course is *about*.")

## 3. The toolchain in four hello-worlds

Four building blocks recur across the course. Here's the smallest possible taste of each — run them,
and when you hit them in a lab you'll recognise the move.

1. **`jax.grad` — the primitive.** Reverse-mode autodiff over a scalar function — and, crucially,
   *through a loop* (a tiny gradient descent). Every fit / optimisation / inverse in this course is
   this, scaled up.
2. **The hypergraph.** A regulon is one *hyperedge*; the clique expansion $A = H H^{\top}$ blows it
   into a quadratic-size gene–gene graph. `hgx.from_incidence(H)` *is* the hypergraph. Labs 2, 4, 5
   live here. (If `hgx` isn't installed, the labs build the clique/star expansions of $H$ by hand —
   so they still run; `uv sync` gives you the real thing.)
3. **A differentiable ODE.** Solve $\dot x = -p\,x$ (here with a 4-line RK4 — exactly the integrator
   Labs 1/5/8/10 use; `diffrax` is the production version with adjoints), then `jax.grad` the solution
   w.r.t. the parameter $p$. Labs 1, 5, 8 are ODEs you differentiate through.
4. **Control.** Build a 2-state plant $\dot x = Ax + Bu$; check Kalman controllability (rank of
   $[B\;AB]$); solve an LQR (`scipy.linalg.solve_continuous_are`). Labs 6, 8 are this — and `jaxctrl`
   is the library version (`is_controllable`, `controllability_gramian`, `lqr`, `minimum_driver_nodes`).

In [ ]:
# ---- (1) jax.grad: the primitive --------------------------------------------------------------
f = lambda x: jnp.sum(jnp.sin(x) ** 2)
x0 = jnp.array([0.3, 1.1, -0.7])
print("(1) jax.grad —  ∇f at x0 =", np.round(np.asarray(jax.grad(f)(x0)), 3))
# ...and *through a loop*: a 50-step gradient descent on g(w) = (w - 2)^2, differentiating the WHOLE thing w.r.t. the start
def descend(w0, lr=0.1, n=50):
    g = lambda w: (w - 2.0) ** 2
    def body(_, w): return w - lr * jax.grad(g)(w)
    return jax.lax.fori_loop(0, n, body, w0)
print("    grad of (final w after 50 GD steps) w.r.t. (initial w):", float(jax.grad(descend)(0.0)), " (≈ how much the start still matters — small, since GD converges)")

# ---- (2) the hypergraph ----------------------------------------------------------------------
Htoy = np.array([[1,1,0,0],[1,0,1,0],[1,0,1,0],[0,1,1,0],[0,0,0,1],[0,0,0,1]], np.float32)  # 6 genes × 4 regulons
print(f"\n(2) hypergraph — toy H ({Htoy.shape[0]} nodes × {Htoy.shape[1]} hyperedges):  node degrees {Htoy.sum(1).astype(int).tolist()}   hyperedge sizes {Htoy.sum(0).astype(int).tolist()}")
print(f"    the size-3 hyperedge (regulon 3) is ONE edge, not a 3-clique — vs the clique expansion A = H Hᵀ which would put {int((Htoy@Htoy.T>0).sum() - Htoy.shape[0])//2} gene–gene edges in.  (Lab 2 unpacks this; the blow-up is the whole point.)")
try:
    import hgx
    hg = hgx.from_incidence(jnp.asarray(Htoy))
    print(f"    hgx.from_incidence(H) — got a {type(hg).__name__}; Labs 2/4/5 use its conv layers (e.g. UniGCNConv) + hodge_laplacians.")
except Exception as e:
    print(f"    [note] hgx not importable ({type(e).__name__}) — Labs 2/4/5 use it (`uv sync` / `pip install hgx`); they also build the clique/star expansions of H by hand, so they still run.")

# ---- (3) a differentiable ODE -----------------------------------------------------------------
def rk4(rhs, x, dt, p):
    k1 = rhs(x, p); k2 = rhs(x + .5*dt*k1, p); k3 = rhs(x + .5*dt*k2, p); k4 = rhs(x + dt*k3, p)
    return x + dt/6.0*(k1 + 2*k2 + 2*k3 + k4)
def solve(p, x0=1.0, T=2.0, n=200):                       # ẋ = -p·x  →  x(T) = x0·exp(-p·T)
    dt = T/n
    return jax.lax.fori_loop(0, n, lambda _, x: rk4(lambda x, p: -p*x, x, dt, p), x0)
print("\n(3) ODE — solve ẋ = -p·x to T=2 with RK4:  x(2) =", float(solve(1.0)), " (exact e^-2 =", float(jnp.exp(-2.0)), ")")
print("    d x(2) / dp  via jax.grad (differentiate *through* the integrator):", float(jax.grad(solve)(1.0)), " (exact -2·e^-2 =", float(-2*jnp.exp(-2.0)), ")")

# ---- (4) control ------------------------------------------------------------------------------
import scipy.linalg as sla
A = np.array([[0.0, 1.0], [-2.0, -3.0]]); B = np.array([[0.0], [1.0]])
ctrb = np.hstack([B, A @ B]); rank = np.linalg.matrix_rank(ctrb)
print(f"\n(4) control — plant ẋ=Ax+Bu:  Kalman ctrb rank = {rank}/2  (controllable = {rank == 2})")
X = sla.solve_continuous_are(A, B, np.eye(2), np.array([[1.0]])); K = (B.T @ X)
print(f"    LQR gain K = {K.ravel().round(3)},  cost-to-go from x=[1,0] = {(np.array([1,0]) @ X @ np.array([1,0])):.3f}   (Labs 6/8; jaxctrl is the library version)")
print("\n✓ if all four printed, you have the toolchain. On to Lab 1.")

## 4. The dependency map, and a note on three "identifiabilities"

**What needs what** (the labs are self-contained but build conceptually):

```
Setup (here) ─┬─► Lab 1  ─┐  (gene-circuit dynamics — skip if you've done Elowitz & Bois)
              │           │
              └─► Lab 2 ──┼─► Lab 3        (fidelity needs the regulome of Lab 2)
                  (regulome)│
                           ├─► Lab 4        (modularity needs the hypergraph of Lab 2)
                           └─► Lab 5 ──┬─► Lab 6 ──┬─► Lab 8 ──► Lab 9
                             (learned ODE)│        │   ▲
                                          │        └── Lab 7 (the symbolic pre-condition; pairs with 6)
                                          └────────────┴──► Lab 10  (cancer needs Lab 4's MII + Lab 5's drivers + Lab 8's compiler)
```
Plus, outside the numbered track: the **`jaxctrl` example notebooks** (the nonlinear / SINDy /
hypergraph-driver worked versions of Labs 6–8); **`organoid_hgx_colab.ipynb`** (the whole real
pipeline on a GPU); and the **BETSE-JAX refactor** at `~/Workspace/betse-unified` (the bioelectric-
layer companion — the partner to Labs 8–10 on the $V_{\rm mem}$ side).

**One word, three meanings — keep them apart** (the course uses all three; this is the disambiguation):

| sense | property of… | the question | tested by | lab |
|---|---|---|---|---|
| **module** identifiability | a network's **topology** | does the regulome split into stable, distinct units? | the Hodge-Laplacian spectral gap → the **Module Identifiability Index** | **4** (and **10**, turned diagnostic) |
| **structural** identifiability | a parametric **dynamic model** | can the rate constants (Hill exponents, …) be recovered from the input–output map, with *perfect, noise-free* data? | symbolic / differential-algebra (Taylor-series / observability-rank; Bellman & Åström; COMBOS) | **7** |
| **practical** identifiability | a model **+ finite, noisy data** | how wide is the posterior / the profile likelihood? | SBI, profile likelihoods, Fisher information | **8** (the SBI inverse) |

And — distinct again — **fidelity** ([Lab 3](03_benchmarking_fidelity.ipynb)): does KO(TF) predict the
screen? A regulome can be a faithful *predictor* and still have *weak module identifiability*; the two
are orthogonal. (A Hypergraph Neural ODE, Lab 5, is by design *structurally non-identifiable* in its
weights — and that's fine, because it's the *flow* you use, not $\theta$.)

## 5. Exercise — break it, then go

**(a) Confirm the environment.** Run the env-check cell and the four hello-worlds. Note which of the
"recommended" packages are missing; install them (`uv sync`) if you want the full experience — though
every lab runs without `jaxctrl` and without `hgx` (it'll print a `[note]` and use a built-in fallback).

**(b) Break one hello-world on purpose.** Comment out `import jax` and re-run hello-world (1); read the
error. Change `solve`'s `rhs` from `-p*x` to `+p*x` (an *unstable* ODE) and watch `x(2)` blow up —
and `jax.grad` follow it. Make `B = [[0],[0]]` in hello-world (4) and watch the Kalman rank drop to 1
(uncontrollable). These are the failure modes the labs run into at scale.

**(c) Pick your start.** If you've worked through Elowitz & Bois' *Biological Circuit Design*, go
straight to **[Lab 2](02_regulomes_and_hypergraphs.ipynb)**. Otherwise do **[Lab 1](01_gene_circuit_dynamics.ipynb)**
first — it's the bridge: the same circuits (Hill functions, the toggle, the repressilator) in this
course's `diffrax`/`jax` toolchain, ending in the "linearise a circuit, then LQR-control it" move that
Labs 6–8 scale up.

**(d) *(optional, for the BETSE-JAX side)*** If you'll be working on the bioelectric companion: clone
`~/Workspace/betse-unified`, read `docs/JAX_REFAC.md`, and run `BETSE_JAX=1 uv run pytest
tests/betse_test/a00_unit/science/test_jax_inverse.py -s` — the verified inverse-design benchmark
(target $V_{\rm mem}$ pattern in → ion currents out, loss → ~1e-10). That `optimize_pattern` is the
bioelectric instance of [Lab 8](08_anatomical_compiler.ipynb); the bridge examples being built on it
(inverse bioelectric design / xenobot motility, $V_{\rm mem}$→GRN prepatterning, morphoceutical
timelines) are the wet-lab partner to [Labs 8–10](08_anatomical_compiler.ipynb).

## Recap & where this goes

- The course = **Setup → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10** — from "a regulon is a hyperedge"
  to "design an intervention that steers a tissue", on one running dataset (the Fleck organoid
  regulome) and one toolchain (**JAX / Equinox / Diffrax + `hgx` / `jaxctrl`**), with a `sympy` detour
  for the symbolic-identifiability pre-condition (Lab 7) and a bioelectric companion (the BETSE-JAX
  refactor) for the $V_{\rm mem}$ layer of the wet-lab programme (Labs 9–10).
- **Prerequisites**: Python + numpy/matplotlib; **JAX** (the differentiable-programming mindset — the
  one that really matters); linear algebra (the graph Laplacian); ODEs / dynamical systems (attractors,
  linear stability); light molecular/dev bio; and — for the gene-circuit-dynamics foundations — Elowitz
  & Bois' *Biological Circuit Design* (with Lab 1 as the bridge), HTGAA Weeks 6–7 for the wet-lab
  context. No prior `hgx` / `jaxctrl` / `diffrax` / `sympy` needed.
- **Next:** [Lab 1 — gene-circuit dynamics in a nutshell](01_gene_circuit_dynamics.ipynb) (the
  bridge from Elowitz & Bois — skip if you've done it), then [Lab 2 — Regulomes and
  hypergraphs](02_regulomes_and_hypergraphs.ipynb).